# 5 · Versions, a package of nodes, the palette

- A newer version names the one it replaces, a `@node` of the same id. The decorated function is the newest. `Upgrade` moves a saved graph on, and `Deprecation` marks a version or a node as going away.
- Each module lists its nodes in `NODES`, so importing it registers nothing.
- `definition.describe()` is the record a frontend renders.

In [ ]:
from typing import Annotated

from conductor import Deprecation, NodeRegistry, Param, Result, Upgrade, node
from conductor.widgets import TextWidget


@node("echo", deprecation=Deprecation(header="Use version 2", migration="Version 2 adds a prefix; existing graphs keep working."))
def echo_v1(text: Annotated[str, Param(title="Input", widget=TextWidget())]) -> Annotated[str, Result(title="Output")]:
    return text


@node("echo", title="Echo", description="Echoes input, with an optional prefix", category="text", replaces=echo_v1, upgrade=Upgrade())
def echo(
    text: Annotated[str, Param(title="Input", widget=TextWidget())],
    prefix: Annotated[str, Param(widget=TextWidget())] = "",
) -> Annotated[str, Result(title="Output")]:
    return f"{prefix}{text}"


registry = NodeRegistry([echo])


## Versions

A graph pinned at version 1 runs version 1. A deprecation is a notice, not an error.

In [ ]:
print("versions:      ", sorted(echo.versions))
print("current:       ", echo.current_version)
print("v1 deprecated? ", echo.versions[1].deprecation)
print("v2 deprecated? ", echo.versions[2].deprecation)
print("v1 inputs:     ", [i.name for i in echo.versions[1].interface.inputs])
print("v2 inputs:     ", [i.name for i in echo.versions[2].interface.inputs])

## Registering a package

```python
from myapp.nodes import math, text

registry = NodeRegistry([*text.NODES, *math.NODES])
```

Below, a throwaway package in a temporary directory.

In [ ]:
import sys
import tempfile
from pathlib import Path

pkg = Path(tempfile.mkdtemp()) / "demo_nodes"
pkg.mkdir()
(pkg / "__init__.py").write_text("")
(pkg / "shout.py").write_text('''
from typing import Annotated

from conductor import Param, node
from conductor.widgets import TextWidget


@node("shout", title="Shout", description="Uppercases and adds a bang", category="text")
def shout(text: Annotated[str, Param(title="Input", widget=TextWidget())]) -> str:
    return text.upper() + "!"


NODES = (shout,)
''')

sys.path.insert(0, str(pkg.parent))
from demo_nodes import shout  # noqa: E402

registry = registry.extended_with(shout.NODES)
print("registered:", list(registry))

## The palette

`describe()` dumps through pydantic: a `dtype` as `{"id": ...}`, a widget as its `kind` and options.

In [ ]:
import json

from conductor import NodeDescription
from pydantic import TypeAdapter

descriptions = [definition.describe() for definition in registry.nodes]
palette = TypeAdapter(list[NodeDescription]).dump_python(descriptions, mode="json")
print(json.dumps(palette[0], indent=2))

## Summary

In [ ]:
print(f"Total nodes: {len(palette)}\n")
for entry in palette:
    print(f"  {entry['id']}: {entry['title']} (current: v{entry['current_version']})")
    for number, version in entry["versions"].items():
        status = "deprecated" if version["deprecation"] else "current" if int(number) == entry["current_version"] else "older"
        print(f"    v{number} ({status})")
        for inp in version["inputs"]:
            optional = " (optional)" if not inp["required"] else ""
            print(f"      in:  {inp['name']}: {inp['dtype']['id']} [{inp['widget']['kind']}]{optional}")
        for out in version["outputs"]:
            print(f"      out: {out['name']}: {out['dtype']['id']}")
    print()